# PEDE — BGE-M3 Embedding Server (Colab + Cloudflare Tunnel)

Layanan **embedding BGE-M3** OpenAI-compatible untuk dipakai backend `nsa` (top-k RAG full-text screening Modul 6).

**Cara pakai:**
1. Menu **Runtime → Change runtime type → Hardware accelerator = GPU (T4)**.
2. Menu **Runtime → Run all**.
3. Tunggu sel terakhir mencetak **PUBLIC URL** + baris `.env`. Salin ke file `.env` repo `nsa`.
4. Biarkan tab Colab tetap terbuka selama dipakai (sesi Colab berhenti bila idle ~90 menit / maks ~12 jam; jalankan ulang untuk URL baru).

> Penting: model & metode (FlagEmbedding `BGEM3FlagModel`, **dense 1024-d**, `max_length=8192`) dibuat **identik** dengan `core/vector_store.py` PEDE, agar vektor query berada di **ruang yang sama** dengan chunk yang sudah tersimpan di Qdrant. Jangan ganti modelnya.

In [ ]:
# 1) Install dependencies (1-3 menit)
!pip install -q FlagEmbedding fastapi "uvicorn[standard]" pycloudflared nest_asyncio requests

In [ ]:
# 2) Muat model BGE-M3 (sama persis dengan PEDE: FlagEmbedding, fp16 di GPU)
import torch
from FlagEmbedding import BGEM3FlagModel

USE_FP16 = torch.cuda.is_available()
print("CUDA tersedia:", USE_FP16, "(pakai GPU agar cepat)")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=USE_FP16)
print("✅ BGE-M3 termuat (dense 1024-d, max_length 8192).")

In [ ]:
# 3) Definisikan API embedding OpenAI-compatible
import secrets
from typing import Union, List
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

# Token untuk melindungi endpoint publik. Backend nsa mengirim 'Authorization: Bearer <token>'.
# Set EMBED_API_KEY di .env nsa = nilai di bawah ini. Kosongkan ("") untuk tanpa auth.
API_KEY = secrets.token_hex(16)

app = FastAPI(title="PEDE BGE-M3 Embedding Server")

class EmbReq(BaseModel):
    input: Union[str, List[str]]
    model: str = "BAAI/bge-m3"

def _embed(texts: List[str]):
    # IDENTIK PEDE: dense_vecs dari BGEM3FlagModel (ruang vektor sama dgn chunk di Qdrant)
    out = model.encode(texts, max_length=8192, return_dense=True,
                       return_sparse=False, return_colbert_vecs=False)
    return [list(map(float, v)) for v in out["dense_vecs"]]

@app.get("/")
def health():
    return {"status": "ok", "model": "BAAI/bge-m3", "dim": 1024}

@app.post("/v1/embeddings")
@app.post("/embeddings")
def embeddings(req: EmbReq, authorization: str = Header(default="")):
    if API_KEY and authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="invalid api key")
    texts = [req.input] if isinstance(req.input, str) else list(req.input)
    vecs = _embed(texts)
    return {
        "object": "list",
        "model": req.model,
        "data": [{"object": "embedding", "index": i, "embedding": v} for i, v in enumerate(vecs)],
        "usage": {"prompt_tokens": 0, "total_tokens": 0},
    }

print("API_KEY:", API_KEY)

In [ ]:
# 4) Jalankan server di thread + buka Cloudflare Tunnel -> URL publik
import threading, time, uvicorn

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(4)

from pycloudflared import try_cloudflare
url = try_cloudflare(port=8000).tunnel

print("\n" + "=" * 64)
print("PUBLIC URL :", url)
print("=" * 64)
print("\nSalin ke .env repo nsa:\n")
print(f"EMBED_ENDPOINT={url}/v1")
print(f"EMBED_API_KEY={API_KEY}")
print("EMBED_MODEL=BAAI/bge-m3")

In [ ]:
# 5) (Opsional) Uji mandiri — harus 200 & dim 1024
import requests
r = requests.post("http://127.0.0.1:8000/v1/embeddings",
                  headers={"Authorization": f"Bearer {API_KEY}"},
                  json={"model": "BAAI/bge-m3", "input": "self-regulated learning in online higher education"})
print("status:", r.status_code)
print("dim:", len(r.json()["data"][0]["embedding"]))

## Setelah dapat URL
1. Tempel 3 baris `EMBED_*` ke `.env` repo **nsa**.
2. Beri tahu Claude (atau jalankan ulang screening) — backend otomatis memakai top-k semantik BGE-M3.
3. Selesai dipakai, hentikan sesi Colab (tunnel & GPU bebas). URL akan kedaluwarsa; jalankan ulang notebook untuk URL baru.

## Penjaga sesi & monitor kesehatan server (jalankan terakhir)
Sel ini menjaga kernel tetap aktif **dan** memverifikasi server embedding benar-benar merespons (bukan sekadar kernel hidup) — menampilkan status, uptime, RAM, GPU, dan PUBLIC URL setiap beberapa menit.

> ⚠️ **Catatan jujur:** ini mengurangi sesi mati karena kernel idle, **tetapi tidak** mencegah Colab memutus sesi akibat tab browser ditutup/tidak aktif (~90 menit) atau batas maksimum ~12 jam. Tetap biarkan tab Colab terbuka. Isi `TG_TOKEN`/`TG_CHAT_ID` (opsional) untuk dapat peringatan Telegram saat server mati/pulih.

In [ ]:
# 7) Penjaga sesi + monitor kesehatan server (biarkan sel ini berjalan)
import time, requests
from datetime import datetime, timezone, timedelta

try:
    import psutil
except ImportError:
    psutil = None

# (Opsional) peringatan Telegram saat server DOWN/pulih. Kosongkan untuk nonaktif.
TG_TOKEN = ""       # mis. "123456:ABC-DEF..."
TG_CHAT_ID = ""     # mis. "123456789"

HEALTH_URL = "http://127.0.0.1:8000/"   # endpoint health server lokal
INTERVAL = 300                          # detik antar pengecekan (5 menit)
WIB = timezone(timedelta(hours=7))      # zona waktu tampilan


def _gpu_stats() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            return f"GPU {(total - free) / 1024**3:.1f}/{total / 1024**3:.1f} GB"
    except Exception:
        pass
    return "GPU n/a"


def _tg(msg: str):
    if not (TG_TOKEN and TG_CHAT_ID):
        return
    try:
        requests.post(f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage",
                      data={"chat_id": TG_CHAT_ID, "text": msg}, timeout=15)
    except Exception as e:
        print("  (gagal kirim Telegram:", e, ")")


public_url = globals().get("url", "(jalankan sel 4 untuk PUBLIC URL)")
print("🛡️  Monitor server BGE-M3 berjalan. Biarkan sel ini aktif & tab tetap terbuka.")
print(f"    PUBLIC URL : {public_url}")
print(f"    Cek tiap {INTERVAL // 60} menit. Tekan tombol ■ (stop) untuk berhenti.\n")

start = time.time()
was_down = False
try:
    while True:
        now = datetime.now(WIB).strftime("%H:%M:%S")
        uptime = timedelta(seconds=int(time.time() - start))
        ram = f"RAM {psutil.virtual_memory().percent:.0f}%" if psutil else "RAM n/a"

        # Verifikasi server benar-benar merespons (bukan hanya kernel hidup)
        try:
            ok = requests.get(HEALTH_URL, timeout=10).status_code == 200
        except Exception:
            ok = False

        status = "🟢 server OK" if ok else "🔴 SERVER DOWN"
        print(f"[{now} WIB] {status} | uptime {uptime} | {ram} | {_gpu_stats()}")

        if not ok and not was_down:
            _tg(f"🔴 PEDE embedding server DOWN ({now} WIB). Periksa Colab.")
            was_down = True
        elif ok and was_down:
            _tg(f"🟢 PEDE embedding server pulih ({now} WIB).")
            was_down = False

        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\n⏹️ Monitor dihentikan.")